In [1]:
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import StatePreparation
from qiskit_aer import AerSimulator

In [2]:
def variance_circuit(amplitudes, C=None):
    amplitudes = np.asarray(amplitudes, dtype=complex)
    amplitudes = amplitudes / np.linalg.norm(amplitudes)

    N = len(amplitudes)
    n = int(np.log2(N))

    if C is None:
        C = 0.5*(N-1)**2
    
    X = QuantumRegister(n, "X")
    Y = QuantumRegister(n, "Y")
    A = QuantumRegister(1, "A")

    qc = QuantumCircuit(X, Y, A)
    
    prep = StatePreparation(amplitudes)

    qc.append(prep, X)
    qc.append(prep, Y)

    qc.barrier()

    # set P(A=1 | i,j) = (i-j)^2/(2C)
    controls = list(X) + list(Y)

    for i in range(N):
        for j in range(N):

            d = i - j

            probability = 0.5 * d**2 / C

            if probability == 0:
                continue

            probability = min(1.0, probability)

            theta = 2 * np.arcsin(np.sqrt(probability))
            
            for k in range(n):
                if ((i >> k) & 1) == 0:
                    qc.x(X[k])

            for k in range(n):
                if ((j >> k) & 1) == 0:
                    qc.x(Y[k])
            
            qc.mcry(
                theta,
                controls,
                A[0]
            )
            
            for k in range(n):
                if ((i >> k) & 1) == 0:
                    qc.x(X[k])
            
            for k in range(n):
                if ((j >> k) & 1) == 0:
                    qc.x(Y[k])
    
    return qc, C

def zero_mean_variance_circuit(
    amplitudes,
    x_values,
    C=None
):
    amplitudes = (amplitudes/np.linalg.norm(amplitudes))

    N = len(amplitudes)
    n = int(np.log2(N))
    
    probabilities = np.abs(amplitudes) ** 2

    mean = np.sum(
        probabilities * x_values
    )

    print("Zero-mean check:")
    print("E[X] =", mean)
    
    max_x2 = np.max(x_values ** 2)

    if C is None:
        C = max_x2
    
    X = QuantumRegister(n, "X")
    A = QuantumRegister(1, "A")

    qc = QuantumCircuit(X, A)
    
    qc.append(
        StatePreparation(amplitudes),
        X
    )

    qc.barrier()

    # Ry(theta_i) -> sin^2(theta_i / 2) = x_i² / C.
    for i in range(N):
        x = x_values[i]

        probability = x**2 / C

        probability = min(
            1.0,
            max(0.0, probability)
        )

        if np.isclose(probability, 0.0):
            continue

        theta = 2 * np.arcsin(
            np.sqrt(probability))
        
        for k in range(n):
            if (i >> k) & 1 == 0:
                qc.x(X[k])
        
        qc.mcry(
            theta,
            list(X),
            A[0]
        )
        
        for k in range(n):
            if (i >> k) & 1 == 0:
                qc.x(X[k])

    return qc, C


def product_std_circuit(psi1, psi2, C1=None, C2=None):
    psi1 = psi1/np.linalg.norm(psi1)
    psi2 = psi2/np.linalg.norm(psi2)
    
    N1 = len(psi1)
    N2 = len(psi2)
    
    n1 = int(np.log2(N1))
    n2 = int(np.log2(N2))
    
    if C1 is None:
        C1 = (N1 - 1) ** 2

    if C2 is None:
        C2 = (N2 - 1) ** 2
    
    X1 = QuantumRegister(n1, "X1")
    X2 = QuantumRegister(n1, "X2")

    Y1 = QuantumRegister(n2, "Y1")
    Y2 = QuantumRegister(n2, "Y2")

    A = QuantumRegister(1, "A")

    qc = QuantumCircuit(
        X1, X2,
        Y1, Y2,
        A
    )
    
    prep1 = StatePreparation(psi1)
    prep2 = StatePreparation(psi2)

    qc.append(prep1, X1)
    qc.append(prep1, X2)

    qc.append(prep2, Y1)
    qc.append(prep2, Y2)

    qc.barrier()

    controls = (
        list(X1)
        + list(X2)
        + list(Y1)
        + list(Y2)
    )

    for x1 in range(N1):
        for x2 in range(N1):
            dx = x1 - x2

            if dx**2 == 0:
                continue

            for y1 in range(N2):
                for y2 in range(N2):

                    dy = y1 - y2
                    if dy**2 == 0:
                        continue

                    probability = (
                        dx**2 * dy**2
                        / (4 * C1 * C2)
                    )

                    probability = min(
                        1.0,
                        max(0.0, probability)
                    )

                    # theta = (x1-x2)^2 (y1-y2)^2/(4 C1 C2)
                    # Ry(theta)|0> -> P(1) = sin²(theta/2)
                    
                    theta = 2 * np.arcsin(
                        np.sqrt(probability)
                    )
                    
                    for k in range(n1):
                        if ((x1 >> k) & 1) == 0:
                            qc.x(X1[k])

                        if ((x2 >> k) & 1) == 0:
                            qc.x(X2[k])

                    for k in range(n2):
                        if ((y1 >> k) & 1) == 0:
                            qc.x(Y1[k])

                        if ((y2 >> k) & 1) == 0:
                            qc.x(Y2[k])

                    qc.mcry(
                        theta,
                        controls,
                        A[0],
                    )
                    
                    for k in range(n1):
                        if ((x1 >> k) & 1) == 0:
                            qc.x(X1[k])

                        if ((x2 >> k) & 1) == 0:
                            qc.x(X2[k])

                    for k in range(n2):
                        if ((y1 >> k) & 1) == 0:
                            qc.x(Y1[k])

                        if ((y2 >> k) & 1) == 0:
                            qc.x(Y2[k])

    return qc, C1, C2max(0.0, probability)

In [3]:
def exact_statistics(amplitudes, x_values):
    amplitudes = amplitudes/np.linalg.norm(amplitudes)
    probabilities = np.abs(amplitudes)**2

    mean = np.sum(probabilities * x_values)

    second_moment = np.sum(probabilities * x_values**2)
    variance = second_moment - mean**2
    std = np.sqrt(max(0.0, variance))

    return mean, second_moment, variance, std

In [4]:
N = 2**6

mu = 1
sigma = 0.1

x_values = np.linspace(
    0,
    1,
    N
)

probabilities = np.exp(-(x_values - mu)**2 / (2*sigma**2))
probabilities /= probabilities.sum()

amplitudes = np.sqrt(probabilities)

mean, second_moment, variance, std = (
    exact_statistics(
        amplitudes,
        x_values
    )
)

print("===================================")
print("CLASSICAL REFERENCE")
print("===================================")

print("Mean              =", mean)
print("E[X^2]            =", second_moment)
print("Variance          =", variance)
print("Std               =", std)

# --------------------------------------------------------
# Quantum circuit
# --------------------------------------------------------



prep = StatePreparation(amplitudes)

qc.append(prep, X)

qc.barrier()

qc_variance, C1, C2 = variance_circuit_one_copy(
    amplitudes,
    x_values
)

print("===================================")
print("CIRCUIT")
print("===================================")

print("C1 =", C1)
print("C2 =", C2)
print("Qubits =", qc.num_qubits)
print("Depth =", qc.depth())

# --------------------------------------------------------
# Measurement
# --------------------------------------------------------

c = ClassicalRegister(2, "c")

qc_measured = qc.copy()
qc_measured.add_register(c)

qc_measured.measure(qc_measured.qubits[-2], c[0])
qc_measured.measure(qc_measured.qubits[-1], c[1])

aer_simulator = AerSimulator()

qc_compiled = transpile(
    qc_measured,
    aer_simulator,
    optimization_level=1
)

shots = int(1E5)

result = aer_simulator.run(
    qc_compiled,
    shots=shots
).result()

counts = result.get_counts()

print("===================================")
print("COUNTS")
print("===================================")

print(counts)

n_mean_1 = 0
n_second_1 = 0

for bitstring, count in counts.items():
    bits = bitstring.replace(" ", "")
    
    second_bit = int(bits[0])
    mean_bit = int(bits[1])

    n_mean_1 += mean_bit*count
    n_second_1 += second_bit*count

p_mean = n_mean_1/shots
p_second = n_second_1/shots

mean_quantum = C1*p_mean
second_quantum = C2*p_second

variance_quantum = (second_quantum - mean_quantum**2)

variance_quantum = max(0.0, variance_quantum)
std_quantum = np.sqrt(variance_quantum)

# --------------------------------------------------------
# Results
# --------------------------------------------------------

print("===================================")
print("QUANTUM RESULT")
print("===================================")

print("P(M=1) =", p_mean)
print("P(S=1) =", p_second)

print("Mean              =", mean_quantum)
print("E[X^2]            =", second_quantum)
print("Variance          =", variance_quantum)
print("Std               =", std_quantum)

# --------------------------------------------------------
# Error
# --------------------------------------------------------

print("===================================")
print("ERROR")
print("===================================")

print(
    "Mean error =",
    abs(mean_quantum - mean)
)

print(
    "E[X^2] error =",
    abs(second_quantum - second_moment)
)

print(
    "Variance error =",
    abs(variance_quantum - variance)
)

print(
    "Std error =",
    abs(std_quantum - std)
)

CLASSICAL REFERENCE
Mean              = 0.9251209346359246
E[X^2]            = 0.8596463390169182
Variance          = 0.0037975953152715736
Std               = 0.06162463237433205


NameError: name 'qc' is not defined